In [1]:
!pip install roboflow

from roboflow import Roboflow

rf = Roboflow(api_key="tWgrSnRgsQE7iL6Sn4Aa")
project = rf.workspace("id-card-vf1zl").project("crop-idcard")
version = project.version(6)
dataset = version.download("coco")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 86.7/86.7 kB 3.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.8/66.8 kB 6.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 20.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 91.8 MB/s eta 0:00:00
  Attempting uninstall: opencv-python-headless
    Found existing installation: opencv-python-headless 4.11.0.86
    Uninstalling opencv-python-headless-4.11.0.86:
      Successfully uninstalled opencv-python-headless-4.11.0.86
  Attempting uninstall: idna
    Found existing installation: idna 3.10
    Uninstalling idna-3.10:
      Successfully uninstalled idna-3.10
loading Roboflow workspace...
loading Roboflow project...



Extracting Dataset Version Zip to Crop-IDCard-6 in coco:: 100%|██████████| 2004/2004 [00:03<00:00, 509.15it/s]


In [11]:
import json
import pandas as pd

coco_path = '/content/Crop-IDCard-6/train/_annotations.coco.json'

angles_df = pd.read_csv("/content/angles.csv")
angles_df["filename"] = angles_df["filename"].str.strip()

with open(coco_path) as f:
    coco = json.load(f)


In [12]:
import shutil

# Copy to current directory (optional)
shutil.copy("/content/Crop-IDCard-6/train/_annotations.coco.json", "/content/annotations.json")


'/content/annotations.json'

In [9]:
data = []

for ann in coco.get("annotations", []):
    img_id = ann['image_id']
    filename = id_to_filename.get(img_id, None)
    if not filename:
        continue

    segmentation = ann.get("segmentation", [])

    # بررسی اینکه segmentation مناسب هست
    if segmentation and isinstance(segmentation[0], list) and len(segmentation[0]) == 8:
        points = segmentation[0]
        corners = [(points[i], points[i+1]) for i in range(0, 8, 2)]

        data.append({
            "filename": filename,
            "top_left": corners[0],
            "top_right": corners[1],
            "bottom_right": corners[2],
            "bottom_left": corners[3]
        })

coco_df = pd.DataFrame(data)
print("✅ تعداد رکوردهای استخراج شده:", len(coco_df))
print(coco_df.head())


✅ تعداد رکوردهای استخراج شده: 0
Empty DataFrame
Columns: []
Index: []


In [8]:
print("🔵 coco_df columns:", coco_df.columns.tolist())


🔵 coco_df columns: []


In [7]:
final_df = pd.merge(angles_df, coco_df, on="filename", how="inner")
final_df.head()

KeyError: 'filename'

In [ ]:
final_df.to_csv("/mnt/data/final_dataset.csv", index=False)